# Lab C — INT8 Quantization on the AMD CPU / INT8 量化
## Slide 39–42 · 更少的位数，更快的计算？

**What will you learn? / 本节学什么？**
- FP32 与 INT8 的区别、为什么需要校准 (calibration) FP32 vs INT8 and why calibration is needed
- 用 NNCF 对 YOLO26l 做训练后量化（真实 COCO 校准图片） Post-training quantization with NNCF on real COCO images
- 同时测量**速度、模型大小、内存、mAP** Measuring speed, size, memory and mAP together

**One key idea / 一个核心原理**
> INT8 用 8 位整数代替 32 位浮点：数据更小、内存带宽更省、且 CPU 有专门的 INT8 指令——但近似会损失精度，必须校准并实测。INT8 replaces fp32 with 8-bit > integers: less data, less bandwidth, dedicated CPU instructions — but the > approximation needs calibration and honest measurement.

In [1]:
from pathlib import Path
import sys
REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
print("repo:", REPO)

repo: /workspace/yolo26-openvino-rocm-day2-labs


## 0. INT8 in five numbers, by hand / 用 5 个数字亲手理解 INT8

先看一个**极简数值例子**，再跑真实模型。假设一层里有 3 个 FP32 权重：
A tiny numeric example first, then the real model. Suppose a layer has 3 FP32 weights:

```python
fp32_values   = [0.00, -0.60, 1.20]     # 原始数值 / original numbers
scale         = 1.20 / 127              # 量化比例: 把最大值映射到 INT127
zero_point    = 0                       # 对称量化: 零点为 0
int8_values   = round(fp32 / scale)     # -> [0, -64, 127]
dequantized   = int8_values * scale     # -> [0.00, -0.605, 1.20]
error         = dequantized - fp32      # -> [0, -0.005, 0]  ← 近似误差!
```

每个 FP32（32 bit）变成 INT8（8 bit）→ **模型小 4 倍**；但 `round()` 丢掉的小数就是**精度误差的来源**——真实模型里数百万个这样的舍入叠加，就可能改变某些检测结果（这就是 ΔmAP）。
Every 32-bit float becomes an 8-bit int → **4x smaller model**; but the rounding error is real — millions of such roundings can flip detections (that is the ΔmAP).


In [2]:
# Run the five-number example / 运行五数字例子
import numpy as np
fp32_values = np.array([0.00, -0.60, 1.20], dtype=np.float32)
scale = float(np.max(np.abs(fp32_values))) / 127.0     # 对称量化 / symmetric
zero_point = 0
int8_values = np.round(fp32_values / scale + zero_point).astype(np.int8)
dequantized = (int8_values.astype(np.float32) - zero_point) * scale
print("fp32  :", fp32_values)
print("scale :", round(scale, 6), "| zero_point:", zero_point)
print("int8  :", int8_values)
print("deq   :", np.round(dequantized, 4))
print("error :", np.round(dequantized - fp32_values, 4), "← 量化误差/quantization error")
print("bits per value: 32 -> 8 (4x smaller)")


fp32  : [ 0.  -0.6  1.2]
scale : 0.009449 | zero_point: 0
int8  : [  0 -64 127]
deq   : [ 0.     -0.6047  1.2   ]
error : [ 0.     -0.0047  0.    ] ← 量化误差/quantization error
bits per value: 32 -> 8 (4x smaller)


## 1. Quantize with real calibration images / 用真实图片校准并量化
校准集来自 COCO **train2017**（150 张，与验证子集**完全不重叠**）。NNCF 会用这些图片统计每一层的数值范围，决定 scale/zero-point。
Calibration images come from COCO **train2017** (150 images, strictly disjoint from the val subset). NNCF uses them to estimate each layer's numeric range → scale/zero-point.

量化缓存带有**完整身份指纹**（模型 SHA、FP32 导出 SHA、校准清单 SHA、校准张数、NNCF 版本……）。
把 `STUDENT_CALIB_N` 从 150 改成 30，系统会**自动生成另一个独立缓存**——绝不会偷偷复用 150 张的旧模型（v2.0.1 修复）。
The INT8 cache carries a **full identity key** (model sha, FP32 export sha, calibration manifest sha, sample count, NNCF version…). Changing `STUDENT_CALIB_N` 150 → 30 automatically builds a **separate cache** — the old 150-image model can never be silently reused (v2.0.1 fix).
首次量化需要几分钟；之后同配置命中缓存。
First quantization takes a few minutes; identical configs then hit the identity-verified cache.


In [3]:
STUDENT_CALIB_N = 150          # <-- 量化校准图片数 (try 30 vs 150)

from course.quantization import quantize_int8, model_dir_size_mb
from course.model import openvino_fp32_dir, openvino_int8_dir
import time

fp32_dir = openvino_fp32_dir()
t0 = time.perf_counter()
int8_dir = quantize_int8(n_calib_images=STUDENT_CALIB_N,
                         subset_size=STUDENT_CALIB_N)
quant_s = time.perf_counter() - t0
print(f"quantization time: {quant_s:.0f} s (cached afterwards)")

[quantize] cache HIT (identity verified): yolo26l_int8_mixed_openvino_model_0ff748ae5747b318 · n_calib=150 preset=mixed
quantization time: 2 s (cached afterwards)


In [4]:
size_fp32 = model_dir_size_mb(fp32_dir)
size_int8 = model_dir_size_mb(int8_dir)
print(f"FP32 IR : {size_fp32:7.1f} MB")
print(f"INT8 IR : {size_int8:7.1f} MB   ({size_fp32/size_int8:.2f}x smaller)")

FP32 IR :    99.9 MB
INT8 IR :    26.4 MB   (3.79x smaller)


## 2. Run the baseline: OpenVINO FP32 / 基线：OpenVINO FP32
同一块 AMD CPU、同一张图片、相同 imgsz/batch，与 Lab B 相同的测量方法。
Same AMD CPU, same image, same measurement method as Lab B.

In [5]:
from ultralytics import YOLO
from course.benchmark import bench, save_run, RunMeta, load_bench_images

WARMUP, ITERS = 3, 10
images = load_bench_images(); IMG = images[0]

fp32_model = YOLO(str(fp32_dir))
fp32_ms = bench(lambda: fp32_model.predict(source=str(IMG), imgsz=640,
                                 device="cpu", verbose=False),
                warmup=WARMUP, iters=ITERS, sync=False)
fp32_sum = save_run(RunMeta("labC_int8", "openvino_fp32", "cpu", "openvino-cpu",
                            precision="fp32", warmup=WARMUP, iters=ITERS,
                            notes="notebook quick run"), fp32_ms)
print(f"FP32 P50 {fp32_sum['stats']['p50_ms']:.1f} ms")

Loading /workspace/yolo26-openvino-rocm-day2-labs/weights/yolo26l_openvino_model for OpenVINO inference...


Using OpenVINO LATENCY mode for batch=1 inference on CPU...


[saved] results/raw/labC_int8/labC_int8_openvino_fp32_20261108_112832.csv  (10 samples)
FP32 P50 46.7 ms


## 3. Change one thing: precision FP32 → INT8 / 只改一件事：精度换成 INT8

In [6]:
int8_model = YOLO(str(int8_dir))
int8_ms = bench(lambda: int8_model.predict(source=str(IMG), imgsz=640,
                                device="cpu", verbose=False),
                warmup=WARMUP, iters=ITERS, sync=False)
int8_sum = save_run(RunMeta("labC_int8", "openvino_int8", "cpu", "openvino-cpu",
                            precision="int8", warmup=WARMUP, iters=ITERS,
                            extra={"calibration_images": STUDENT_CALIB_N}), int8_ms)
print(f"INT8 P50 {int8_sum['stats']['p50_ms']:.1f} ms")

Loading /workspace/yolo26-openvino-rocm-day2-labs/weights/int8_cache/yolo26l_int8_mixed_openvino_model_0ff748ae5747b318 for OpenVINO inference...


Using OpenVINO LATENCY mode for batch=1 inference on CPU...


[saved] results/raw/labC_int8/labC_int8_openvino_int8_20261108_112836.csv  (10 samples)
INT8 P50 23.4 ms


## 4. Compare: speed AND accuracy / 对比：速度与精度
量化永远要问两个问题：**多快？多准？** 只报速度不报精度是不诚实的。
Quantization must answer two questions: *how fast* and *how accurate*. Reporting speed without accuracy is dishonest.

In [7]:
from course.benchmark import map5095

map_fp32 = map5095(fp32_model, str(REPO / "data/val_subset.yaml"), 640, 8, "cpu")
map_int8 = map5095(int8_model, str(REPO / "data/val_subset.yaml"), 640, 8, "cpu")
import pandas as pd
sp = fp32_sum["stats"]["p50_ms"] / int8_sum["stats"]["p50_ms"]
display(pd.DataFrame([
    {"model": "FP32", "size (MB)": round(size_fp32, 0), "P50 (ms)": round(fp32_sum['stats']['p50_ms'], 1),
     "speedup": "1.00x", "mAP50-95": round(map_fp32['map50_95'], 4)},
    {"model": "INT8", "size (MB)": round(size_int8, 0), "P50 (ms)": round(int8_sum['stats']['p50_ms'], 1),
     "speedup": f"{sp:.2f}x", "mAP50-95": round(map_int8['map50_95'], 4)}]))

Ultralytics 8.4.207 🚀 Python-3.12.3 torch-2.12.0+rocm7.14.1 CPU (AMD EPYC 9334 32-Core Processor)


Loading /workspace/yolo26-openvino-rocm-day2-labs/weights/yolo26l_openvino_model for OpenVINO inference...


Using OpenVINO LATENCY mode for batch=1 inference on CPU...


Setting batch=1 input of shape (1, 3, 640, 640)


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2027.3±566.9 MB/s, size: 166.3 KB)


val: Scanning /workspace/yolo26-openvino-rocm-day2-labs/data/val_subset/labels.cache... 120 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 120/120 14.4Mit/s 0.0s

... (108 outputs left out)

                   all        120        807      0.807      0.634      0.769      0.568


Speed: 0.5ms preprocess, 35.7ms inference, 0.0ms loss, 1.9ms postprocess per image


Results saved to /workspace/yolo26-openvino-rocm-day2-labs/runs/detect/val-21


,model,size (MB),P50 (ms),speedup,mAP50-95
0,FP32,100.0,46.7,1.00x,0.6349
1,INT8,26.0,23.4,2.00x,0.5682


In [8]:
from course.charts import chart_fp32_vs_int8
chart_fp32_vs_int8(fp32_sum, int8_sum, size_fp32, size_int8,
                   map_fp32["map50_95"], map_int8["map50_95"])

[chart] /workspace/yolo26-openvino-rocm-day2-labs/assets/slides/03_fp32_vs_int8.png (+svg)


## 5. Why? / 为什么会这样？
- 模型大小 ≈ 4× 缩小：每个权重 32 bit → 8 bit，这是确定的。
- 速度取决于 CPU 是否有高效 INT8 路径。本机 EPYC 9334 (Zen4) 有 AVX-512 VNNI 但**没有 AMX**；oneDNN 的 INT8 卷积是否比 FP32 快、快多少，由实测决定。
- 精度损失取决于数值分布与校准质量——ΔmAP 可能小到可忽略，也可能不可接受。

- Size ≈ 4× smaller is guaranteed (32→8 bits per weight).
- Speed depends on the CPU's INT8 path. This EPYC 9334 (Zen 4) has AVX-512 VNNI but **no AMX**; whether oneDNN INT8 beats FP32 here is decided by the measurement above, not by marketing.
- Accuracy loss depends on calibration — ΔmAP must be checked, never assumed.

**INT8 是否更快，本实验的答案就是上面表格里的数字。**

## 6. Try it yourself — and answer the deployment question / 学生练习与部署之问

**先预测再验证**：把三个预测写进变量；然后回到第 1 格把 `STUDENT_CALIB_N` 改成 `30`（会自动重新量化并生成**独立缓存**，可与 150 的结果并存比较）。
Predict size/speed/accuracy first; then set `STUDENT_CALIB_N = 30` — it re-quantizes automatically into a SEPARATE cache, coexisting with the 150-image model for comparison.


In [9]:
my_size_prediction   = "INT8 size will be ___ MB"     # <-- write me
my_speed_prediction  = "INT8 P50 will be ___ ms"     # <-- write me
my_acc_prediction    = "ΔmAP will be ___"            # <-- write me
for s in (my_size_prediction, my_speed_prediction, my_acc_prediction):
    print("prediction:", s)
print()
print(f"measured size : {size_int8:.1f} MB")
print(f"measured P50  : {int8_sum['stats']['p50_ms']:.1f} ms")
print(f"measured ΔmAP : {map_int8['map50_95'] - map_fp32['map50_95']:+.4f}")

prediction: INT8 size will be ___ MB
prediction: INT8 P50 will be ___ ms
prediction: ΔmAP will be ___

measured size : 26.4 MB
measured P50  : 23.4 ms
measured ΔmAP : -0.0667


## 7. The deployment question / 部署之问（必须回答）

INT8 更快更小，但 mAP 降低了。**你觉得值得部署吗？** 没有标准答案——取决于业务：
INT8 is faster and smaller but loses mAP. **Is it worth deploying?** There is no universal answer — it depends:
- 安防漏检一个人的代价 vs 服务器成本减半？*Cost of a missed detection vs halved server cost?*
- 延迟要求是否必须 INT8 才能达到？*Is INT8 required to meet the latency SLO?*
- 能否接受 QAT（量化感知训练）的额外训练成本换取更小 ΔmAP？*Would QAT's extra training cost buy a smaller ΔmAP?*

**不要形成"位数越低越好"的直觉**——精度代价必须与收益一起呈报。
**Never conclude "lower bits are always better"** — the accuracy price must be reported with the gain.


## Common mistakes / 常见错误
- 把量化说成"精度无损"：本实验实测 ΔmAP 是负数，必须如实报告。
  *Never say "lossless quantization"*: our measured ΔmAP is negative — report it.
- 校准集与验证集重叠：会造成精度虚高。本课程两者来自不同 COCO split。
  *Calibration/val overlap inflates accuracy*: ours come from different COCO splits.
- 期望 INT8 一定更快：没有 AMX 的 AMD CPU 上，INT8 是否提速由 oneDNN 内核与内存带宽决定，必须实测。
  *INT8 is not guaranteed faster*: without AMX, whether it helps depends on oneDNN kernels and bandwidth — measure it.


**Takeaway / 一句话总结**
> 更少的位数可能带来更小的模型和更低的计算成本——但加速取决于硬件指令支持，精度代价必须一起测量。
> Lower precision can shrink models and cut cost — but speedup depends on > the hardware, and the accuracy price must always be measured alongside.